In [29]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [30]:
df_path = "dataset"

In [31]:
data_set = df_path

In [32]:
import os
all_data = os.listdir(data_set)

In [33]:
all_data[:5]

['201311077', '201311192', '213311023', '221311076', '221311105']

In [34]:
len(all_data)

47

In [35]:
import os

# Get unique extensions from the dataset directory
extensions = set()
for root, dirs, files in os.walk(data_set):
    for file in files:
        extensions.add(os.path.splitext(file)[1].lower())

print(f"Unique file types found: {extensions}")

Unique file types found: {'.jpeg', '.dng', '.heic', '.jpg'}


In [36]:
from collections import Counter
import os

# count images types
exts = ['.dng', '.jpg', '.heic', '.jpeg']
counts = Counter(os.path.splitext(f)[1].lower() for r, d, files in os.walk(data_set) for f in files if os.path.splitext(f)[1].lower() in exts)

print(f"Counts: {dict(counts)}\nTotal: {sum(counts.values())}")

Counts: {'.jpg': 133, '.jpeg': 98, '.dng': 3, '.heic': 4}
Total: 238


In [37]:
import os
import cv2
import numpy as np
import random
from PIL import Image, ImageEnhance
from pillow_heif import register_heif_opener
from skimage.morphology import skeletonize

In [38]:
register_heif_opener()

In [39]:

RAW_DATASET = "dataset"
OUTPUT_DATASET = "dataset_v2"

CONVERTIBLE_EXTS = {'.png', '.dng', '.heic', '.tiff', '.bmp', '.jpeg', '.gif', '.webp', '.tif', '.jpg'}
AUGMENT_SUFFIXES = ('_rot', '_trans', '_noise', '_sharp',
                    '_laplacian', '_gabor', '_morph_opening', '_morph_gradient', '_skeleton')

# image Processing

In [40]:
def _to_gray(img_bgr):
    return img_bgr if img_bgr.ndim == 2 else cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

def apply_laplacian(img):
    return cv2.convertScaleAbs(cv2.Laplacian(_to_gray(img), cv2.CV_64F))

def apply_gabor(img):
    gray = _to_gray(img)
    responses = [cv2.filter2D(gray, cv2.CV_32F,
                     cv2.getGaborKernel((21, 21), 5.0, t, 10.0, 0.5, 0, ktype=cv2.CV_32F))
                 for t in (0, np.pi/4, np.pi/2, 3*np.pi/4)]
    return cv2.convertScaleAbs(np.max(np.stack(responses), axis=0))

def apply_morph_opening(img):
    return cv2.morphologyEx(_to_gray(img), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

def apply_morph_gradient(img):
    return cv2.morphologyEx(_to_gray(img), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))

def apply_skeleton(img):
    gray = _to_gray(img)
    return (skeletonize(gray < 245).astype(np.uint8) * 255)

FILTERS = {
    'laplacian':     apply_laplacian,      # edge sharpness
    'gabor':         apply_gabor,          # texture / stroke orientation
    'morph_opening': apply_morph_opening,  # noise removal
    'morph_gradient':apply_morph_gradient, # stroke boundary
    'skeleton':      apply_skeleton,       # pen stroke thinning
}


# Data Augmentation Techniques 


In [41]:
def augment(img_pil, name, out_dir):
    img_pil = img_pil.convert('RGB')
    w, h = img_pil.size

    # 1. Random Rotation
    img_pil.rotate(random.randint(-30, 30)).save(
        os.path.join(out_dir, f"{name}_rot.jpg"), 'JPEG')

    # 2. Random Horizontal Translation
    shift = random.randint(-w // 5, w // 5)
    img_pil.transform(img_pil.size, Image.AFFINE, (1, 0, shift, 0, 1, 0)).save(
        os.path.join(out_dir, f"{name}_trans.jpg"), 'JPEG')

    # 3. Gaussian Noise Injection
    arr = np.array(img_pil).astype(np.float64)
    noisy = Image.fromarray(np.clip(arr + np.random.normal(0, 25, arr.shape), 0, 255).astype(np.uint8))
    noisy.save(os.path.join(out_dir, f"{name}_noise.jpg"), 'JPEG')

    # 4. Random Sharpness Adjustment
    ImageEnhance.Sharpness(img_pil).enhance(random.uniform(0.5, 2.0)).save(
        os.path.join(out_dir, f"{name}_sharp.jpg"), 'JPEG')

# Define Pipeline

In [42]:
print(f"Pipeline: {RAW_DATASET}  →  convert  →  5 filters  →  4 augmentations  →  {OUTPUT_DATASET}\n")

total_src = total_filter = total_aug = 0

for root, _, files in os.walk(RAW_DATASET):
    rel = os.path.relpath(root, RAW_DATASET)
    out_dir = os.path.join(OUTPUT_DATASET, rel)
    os.makedirs(out_dir, exist_ok=True)

    for f in files:
        ext = os.path.splitext(f)[1].lower()
        if ext not in CONVERTIBLE_EXTS:
            continue

        base = os.path.splitext(f)[0]
        if any(base.endswith(s) for s in AUGMENT_SUFFIXES):
            continue

        src_path = os.path.join(root, f)
        jpg_path = os.path.join(out_dir, f"{base}.jpg")

        # Step 1: Convert to JPG
        if not os.path.exists(jpg_path):
            try:
                with Image.open(src_path) as img:
                    img.convert('RGB').save(jpg_path, 'JPEG', quality=95)
            except Exception as e:
                print(f"[error] {f}: {e}")
                continue
        total_src += 1

        # Step 2: Image Processing (5 filters)
        cv_img = cv2.imread(jpg_path)
        if cv_img is None:
            print(f"[error] cannot read {jpg_path}")
            continue

        for fname, fn in FILTERS.items():
            filt_path = os.path.join(out_dir, f"{base}_{fname}.jpg")
            if not os.path.exists(filt_path):
                cv2.imwrite(filt_path, fn(cv_img.copy()))
            total_filter += 1  # Count all filters, not just newly created

        # Step 3: Augmentation (4 variants)
        aug_paths = [os.path.join(out_dir, f"{base}_{suffix.lstrip('_')}.jpg") 
                     for suffix in ('_rot', '_trans', '_noise', '_sharp')]
        if not all(os.path.exists(p) for p in aug_paths):
            with Image.open(jpg_path) as pil_img:
                augment(pil_img, base, out_dir)
        total_aug += 4  # Each image generates 4 augmentations

print(f"\nDone.")
print(f"  Source images      : {total_src}")
print(f"  Filter outputs     : {total_filter}  (5 per image)")
print(f"  Augmented sets     : {total_aug // 4} images × 4 variants = {total_aug}")
print(f"  Total output files : {total_src + total_filter + total_aug}")
print(f"  Output folder      : {OUTPUT_DATASET}")


Pipeline: dataset  →  convert  →  5 filters  →  4 augmentations  →  dataset_v2


Done.
  Source images      : 238
  Filter outputs     : 1190  (5 per image)
  Augmented sets     : 238 images × 4 variants = 952
  Total output files : 2380
  Output folder      : dataset_v2
